# 05 — Master-table data and leakage audit

Run after Notebook 03 and before walk-forward model training. **Read-only**: no refitting or overwriting saved artifacts.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from stock_ml_dynamic.data_audit import audit_saved_master
from stock_ml_dynamic.pooled_models import load_master

ROOT = Path("artifacts/master_tables/1d")
summary, missing_latest, missing_features, violations = audit_saved_master(ROOT)
display(pd.Series(summary, name="value").to_frame())


## Missing latest-bar stocks and feature values

An incomplete latest row is retained for investigation, but excluded from prediction. The total stock count comes from the saved master table; compare against the live PostgreSQL universe separately.


In [ ]:
display(missing_latest)
display(missing_features.query("latest_missing > 0 or master_missing > 0"))
assert not violations, f"Audit violations: {violations}"


## Chronological split and target alignment

A row is train-eligible only if its *next-bar timestamp* is no later than the cutoff. Quantile edges, candle K-means, and feature preprocessing must be fit using the training period. This notebook checks saved state, but **cannot independently prove fitting provenance**; walk-forward validation must refit all learned transformations inside each fold.


In [ ]:
master, latest, bundle, manifest = load_master(ROOT)
cutoff = pd.Timestamp(bundle["cutoff"])
train = master["next_bar_timestamp"] <= cutoff
test = master["bar_timestamp"] > cutoff
print("Cutoff:", cutoff)
print("Train rows:", train.sum(), "Test rows:", test.sum())
print("Max training outcome timestamp:", master.loc[train,"next_bar_timestamp"].max())
print("Min test feature timestamp:", master.loc[test,"bar_timestamp"].min())
print("Quantile edges:", bundle["quantiles"].edges_)
print("Candle clusters:", bundle["clusters"].chosen_k_)
print("Encoded columns:", len(bundle["preprocessing"].get_feature_names_out()))


## Inspect one stock: target shift

The direction on timestamp *t* labels the return from *t* to the next available candle for the same ticker.


In [ ]:
symbol = "AAPL"
cols = ["symbol","bar_timestamp","adj_close","next_bar_timestamp","next_adj_close","next_return","direction"]
display(master.loc[master.symbol.eq(symbol),cols].tail(8))
display(latest.loc[latest.symbol.eq(symbol),cols])


## Check the actual database universe (optional)

This query verifies whether a ticker is missing entirely from saved outputs. It requires local PostgreSQL access.


In [ ]:
from sqlalchemy import text
from stock_ml_dynamic.db import engine, qualified_table
schema, table = qualified_table()
with engine().connect() as conn:
    universe = pd.read_sql(text(f'SELECT symbol, COUNT(*) AS bars FROM "{schema}"."{table}" WHERE interval_code=:interval GROUP BY symbol ORDER BY symbol'),conn,params={"interval": "1d"})
saved_symbols = set(master.symbol).union(set(latest.symbol))
print("Database symbols:", len(universe), "Saved symbols:", len(saved_symbols))
print("Missing entirely from saved tables:", sorted(set(universe.symbol)-saved_symbols))
print("Saved symbols absent from database:", sorted(saved_symbols-set(universe.symbol)))


## Next step

Share the summary, `missing_latest`, and database symbol comparison. We will fix any anomalies before implementing fold-specific, leakage-safe walk-forward training.
